# Paper PoRT WMDP Pipeline Smoke Test

This notebook is the first smoke test for the paper-style PoRT WMDP pipeline. It runs a tiny `max_samples` job through the IPC -> post-judgment -> rethink flow from `PoRT_pipeline/WMDP/port_pipeline_wmdp.py`.

It intentionally requires real PoRT artifacts. It does not use the earlier classifier-gated corruption-hook notebooks as a substitute. If an artifact is missing, the notebook fails early with a concrete list of required env vars or URLs.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
POST_CLASSIFIER_DIR = PROJECT_ROOT / 'post_classifier' / 'train'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
for path in [ECO_ROOT, POST_CLASSIFIER_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'peft': 'peft',
    'llm2vec': 'llm2vec',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This PoRT smoke test is intended for a Kaggle GPU runtime.')
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')


## Runtime Config

Set artifact env vars in Kaggle before running if they are not public Hugging Face repo IDs.

Required for a real PoRT smoke:
- `PORT_T5_MODEL_PATH` or `PORT_T5_MODEL_HF_REPO`
- `PORT_CLASSIFIER_BASE_MODEL`
- `PORT_CLASSIFIER_HEAD_CKPT` or `PORT_CLASSIFIER_HEAD_URL`

Defaults keep the target model small enough for Kaggle T4 smoke testing: `microsoft/phi-1_5`, one WMDP domain, and `max_samples=2`.


In [ ]:
def env_text(name, default=None):
    value = os.environ.get(name)
    if value is None:
        return default
    value = value.strip()
    return value if value else default

TARGET_MODEL_HUB_NAME = env_text('PORT_TARGET_MODEL_HUB_NAME', 'microsoft/phi-1_5')
TARGET_MODEL_PATH = env_text('PORT_TARGET_MODEL_PATH', TARGET_MODEL_HUB_NAME)
MODEL_NAME = env_text('PORT_TARGET_CONFIG_NAME', 'phi-1_5')
TORCH_DTYPE = env_text('PORT_TORCH_DTYPE', 'float16')

T5_MODEL_PATH = env_text('PORT_T5_MODEL_PATH') or env_text('PORT_T5_MODEL_HF_REPO')
T5_MODEL_URL = env_text('PORT_T5_MODEL_URL')
CLASSIFIER_BASE_MODEL = env_text('PORT_CLASSIFIER_BASE_MODEL')
CLASSIFIER_HEAD_CKPT = env_text('PORT_CLASSIFIER_HEAD_CKPT')
CLASSIFIER_HEAD_URL = env_text('PORT_CLASSIFIER_HEAD_URL')

WMDP_VARIANT = env_text('PORT_WMDP_VARIANT', 'composite')
WMDP_DOMAIN = env_text('PORT_WMDP_DOMAIN', 'bio')
MAX_SAMPLES = int(env_text('PORT_MAX_SAMPLES', '2'))
BATCH_SIZE = int(env_text('PORT_BATCH_SIZE', '1'))
ICL_EXAMPLE_K = int(env_text('PORT_ICL_EXAMPLE_K', '3'))
CLASSIFIER_CONF_THRESHOLD = float(env_text('PORT_CLASSIFIER_CONF_THRESHOLD', '0.97'))
DEVICE = env_text('PORT_DEVICE', 'cuda:0')

RUN_NAME = env_text('PORT_RUN_NAME', f'paper_port_wmdp_smoke_{WMDP_VARIANT}_{WMDP_DOMAIN}_{MODEL_NAME}')
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
RUN_DIR = OUTPUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'run_name': RUN_NAME,
    'target_model_hub_name': TARGET_MODEL_HUB_NAME,
    'target_model_path': TARGET_MODEL_PATH,
    'model_name': MODEL_NAME,
    'torch_dtype': TORCH_DTYPE,
    't5_model_path': T5_MODEL_PATH,
    't5_model_url': T5_MODEL_URL,
    'classifier_base_model': CLASSIFIER_BASE_MODEL,
    'classifier_head_ckpt': CLASSIFIER_HEAD_CKPT,
    'classifier_head_url': CLASSIFIER_HEAD_URL,
    'wmdp_variant': WMDP_VARIANT,
    'wmdp_domain': WMDP_DOMAIN,
    'max_samples': MAX_SAMPLES,
    'batch_size': BATCH_SIZE,
    'run_dir': str(RUN_DIR),
}, indent=2))


In [ ]:
import urllib.request
import zipfile
import tarfile

VALID_VARIANTS = {'original', 'noise_prefix', 'composite'}
VALID_DOMAINS = {'bio', 'chem', 'cyber'}
DOMAIN_TO_SET = {'bio': 'wmdp-bio', 'chem': 'wmdp-chem', 'cyber': 'wmdp-cyber'}
if WMDP_VARIANT not in VALID_VARIANTS:
    raise ValueError(f'Unsupported WMDP_VARIANT={WMDP_VARIANT}; expected one of {sorted(VALID_VARIANTS)}')
if WMDP_DOMAIN not in VALID_DOMAINS:
    raise ValueError(f'Unsupported WMDP_DOMAIN={WMDP_DOMAIN}; expected one of {sorted(VALID_DOMAINS)}')
if MAX_SAMPLES <= 0:
    raise ValueError('PORT_MAX_SAMPLES must be positive for smoke testing.')

pipeline_script_path = PROJECT_ROOT / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py'
example_library_path = PROJECT_ROOT / 'dataset' / 'AST' / 'demonstrations.json'
eco_config_path = ECO_ROOT / 'config' / 'model_config'


def download_to(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading {url} -> {destination}')
    urllib.request.urlretrieve(url, destination)
    return destination


def maybe_download_archive(url, destination_dir):
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    archive_path = destination_dir / Path(url.split('?')[0]).name
    download_to(url, archive_path)
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(destination_dir)
        return destination_dir
    if tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as tf:
            tf.extractall(destination_dir)
        return destination_dir
    return archive_path

if T5_MODEL_PATH is None and T5_MODEL_URL:
    T5_MODEL_PATH = str(maybe_download_archive(T5_MODEL_URL, RUN_DIR / 't5_model'))
if CLASSIFIER_HEAD_CKPT is None and CLASSIFIER_HEAD_URL:
    CLASSIFIER_HEAD_CKPT = str(download_to(CLASSIFIER_HEAD_URL, RUN_DIR / 'classifier_head' / Path(CLASSIFIER_HEAD_URL.split('?')[0]).name))

missing = []
if not pipeline_script_path.exists():
    missing.append(f'pipeline script not found: {pipeline_script_path}')
if not POST_CLASSIFIER_DIR.exists():
    missing.append(f'post classifier source not found: {POST_CLASSIFIER_DIR}')
if not example_library_path.exists():
    missing.append(f'example library not found: {example_library_path}')
if T5_MODEL_PATH is None:
    missing.append('Set PORT_T5_MODEL_PATH, PORT_T5_MODEL_HF_REPO, or PORT_T5_MODEL_URL')
if CLASSIFIER_BASE_MODEL is None:
    missing.append('Set PORT_CLASSIFIER_BASE_MODEL')
if CLASSIFIER_HEAD_CKPT is None:
    missing.append('Set PORT_CLASSIFIER_HEAD_CKPT or PORT_CLASSIFIER_HEAD_URL')
elif not Path(CLASSIFIER_HEAD_CKPT).exists():
    missing.append(f'classifier head checkpoint not found: {CLASSIFIER_HEAD_CKPT}')

artifact_audit = {
    'pipeline_script_path': str(pipeline_script_path),
    'post_classifier_dir': str(POST_CLASSIFIER_DIR),
    'eco_root': str(ECO_ROOT),
    'eco_config_path': str(eco_config_path),
    'example_library_path': str(example_library_path),
    't5_model_path': T5_MODEL_PATH,
    'classifier_base_model': CLASSIFIER_BASE_MODEL,
    'classifier_head_ckpt': CLASSIFIER_HEAD_CKPT,
    'missing': missing,
}
(RUN_DIR / 'artifact_audit.json').write_text(json.dumps(artifact_audit, indent=2, default=str), encoding='utf-8')
print(json.dumps(artifact_audit, indent=2, default=str))

if missing:
    raise RuntimeError(
        'Missing required PoRT paper artifacts. Provide them via env vars/URLs; see artifact_audit.json. Missing: ' +
        '; '.join(missing)
    )


## Runtime Patch Of Paper Pipeline

The checked-in WMDP paper script still contains local placeholders and one stale model-key reference. This cell writes a runtime copy under the run directory with only path/runtime fixes, leaving the IPC/post-judgment/rethink functions intact.


In [ ]:
import importlib.util
import ast

source = pipeline_script_path.read_text(encoding='utf-8')
source = source.replace('POST_CLASSIFIER_DIR = "{PATH_PLACEHOLDER}"', f'POST_CLASSIFIER_DIR = r"{POST_CLASSIFIER_DIR}"')
source = source.replace('ECO_DIR = "{PATH_PLACEHOLDER}"', f'ECO_DIR = r"{ECO_ROOT}"')
source = source.replace('torch_dtype=torch.bfloat16', 'torch_dtype=getattr(torch, args.torch_dtype)')
source = source.replace('wrapped_model = WrappedModel(models["llama_model"], models["llama_tokenizer"])', 'wrapped_model = WrappedModel(models["main_llama_model"], models["llama_tokenizer"])')

runtime_script_path = RUN_DIR / 'runtime_port_pipeline_wmdp.py'
runtime_script_path.write_text(source, encoding='utf-8')
ast.parse(source, filename=str(runtime_script_path))

spec = importlib.util.spec_from_file_location('runtime_port_pipeline_wmdp', runtime_script_path)
port_wmdp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_wmdp)
print(f'Loaded runtime PoRT module: {runtime_script_path}')


## Build WMDP Smoke Prompts

For `original`, prompts are assembled from `question + choices`. For `noise_prefix` and `composite`, prompts use the preformatted `full_question` column, matching the validated baseline notebook.


In [ ]:
from datasets import Dataset, DatasetDict, load_from_disk
import pandas as pd

WMDP_SET = DOMAIN_TO_SET[WMDP_DOMAIN]
SUBJECT = WMDP_DOMAIN
CHOICE_LABELS = ['A', 'B', 'C', 'D']

def normalize_choices(choices):
    if hasattr(choices, 'tolist'):
        choices = choices.tolist()
    return [str(choice).strip() for choice in choices]

def format_original_prompt(question, choices):
    lines = [str(question)]
    for idx, choice in enumerate(choices):
        lines.append(f'{chr(65 + idx)}. {choice}')
    lines.append('Answer with only the letter:')
    return '\n'.join(lines)

def load_wmdp_split(variant, domain):
    if variant == 'original':
        parquet_path = PROJECT_ROOT / 'dataset' / 'WMDP' / 'original' / f'wmdp-{domain}' / 'test-00000-of-00001.parquet'
        if not parquet_path.exists():
            raise FileNotFoundError(parquet_path)
        dataset = Dataset.from_parquet(str(parquet_path))
        prompt_source = 'question_plus_choices'
    else:
        dataset_path = PROJECT_ROOT / 'dataset' / 'WMDP' / variant / domain
        if not dataset_path.exists():
            raise FileNotFoundError(dataset_path)
        loaded = load_from_disk(str(dataset_path))
        dataset = loaded['test'] if isinstance(loaded, DatasetDict) else loaded
        prompt_source = 'full_question'
    return dataset, prompt_source

dataset, prompt_source = load_wmdp_split(WMDP_VARIANT, WMDP_DOMAIN)
if MAX_SAMPLES > 0:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

records = []
for idx, row in enumerate(dataset):
    choices = normalize_choices(row['choices'])
    if prompt_source == 'full_question':
        prompt = str(row['full_question'])
    else:
        prompt = format_original_prompt(row['question'], choices)
    records.append({
        'row_index': idx,
        'question': row.get('question', ''),
        'prompt': prompt,
        'choices': choices,
        'correct_answer_index': int(row['answer']),
        'correct_answer_text': choices[int(row['answer'])] if 0 <= int(row['answer']) < len(choices) else '',
    })

if not records:
    raise RuntimeError('No WMDP records selected for smoke test.')

prompt_preview = records[0]['prompt'][:800]
print(json.dumps({
    'variant': WMDP_VARIANT,
    'domain': WMDP_DOMAIN,
    'wmdp_set': WMDP_SET,
    'rows': len(records),
    'prompt_source': prompt_source,
    'first_prompt_preview': prompt_preview,
}, indent=2, ensure_ascii=False))

if WMDP_VARIANT in {'noise_prefix', 'composite'}:
    assert prompt_source == 'full_question'
    assert records[0]['prompt'].rstrip().endswith('Answer:')
else:
    assert prompt_source == 'question_plus_choices'


## Run PoRT Smoke

This loads the T5 IPC model, target model, and post classifier, then runs the paper pipeline functions on the selected few prompts.


In [ ]:
from types import SimpleNamespace
import time

with open(example_library_path, 'r', encoding='utf-8') as f:
    example_library = json.load(f)

args = SimpleNamespace(
    t5_model_path=T5_MODEL_PATH,
    model_path=TARGET_MODEL_PATH,
    model_hub_name=TARGET_MODEL_HUB_NAME,
    eco_config_path=str(eco_config_path),
    model_name=MODEL_NAME,
    wmdp_set=WMDP_SET,
    classifier_base_model=CLASSIFIER_BASE_MODEL,
    classifier_head_ckpt=CLASSIFIER_HEAD_CKPT,
    example_library_path=str(example_library_path),
    example_library=example_library,
    output_dir=str(RUN_DIR / 'port_outputs'),
    icl_example_k=ICL_EXAMPLE_K,
    classifier_conf_threshold=CLASSIFIER_CONF_THRESHOLD,
    batch_size=BATCH_SIZE,
    eval_batch_size=BATCH_SIZE,
    max_samples=MAX_SAMPLES,
    device=DEVICE,
    torch_dtype=TORCH_DTYPE,
)

run_config = {
    'purpose': 'paper_port_wmdp_smoke',
    'project_root': str(PROJECT_ROOT),
    'commit': commit_sha,
    'runtime_script_path': str(runtime_script_path),
    'target_model_hub_name': TARGET_MODEL_HUB_NAME,
    'target_model_path': TARGET_MODEL_PATH,
    'model_name': MODEL_NAME,
    'torch_dtype': TORCH_DTYPE,
    't5_model_path': T5_MODEL_PATH,
    'classifier_base_model': CLASSIFIER_BASE_MODEL,
    'classifier_head_ckpt': CLASSIFIER_HEAD_CKPT,
    'wmdp_variant': WMDP_VARIANT,
    'wmdp_domain': WMDP_DOMAIN,
    'wmdp_set': WMDP_SET,
    'prompt_source': prompt_source,
    'max_samples': MAX_SAMPLES,
    'batch_size': BATCH_SIZE,
    'classifier_conf_threshold': CLASSIFIER_CONF_THRESHOLD,
    'run_dir': str(RUN_DIR),
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2, default=str), encoding='utf-8')

start_load = time.perf_counter()
models = port_wmdp.setup_all_models(args)
model_load_seconds = time.perf_counter() - start_load

tokenizer = models['llama_tokenizer']
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if getattr(tokenizer, 'chat_template', None) is None:
    tokenizer.chat_template = "{% for message in messages %}{{ message['content'] }}{% if not loop.last %}\n{% endif %}{% endfor %}{% if add_generation_prompt %}\n{% endif %}"

prompts = [item['prompt'] for item in records]
start_run = time.perf_counter()
generated_answers, rethink_count, rethink_info, final_generation_prompts = port_wmdp.run_end_to_end_for_questions(prompts, models, args)
run_seconds = time.perf_counter() - start_run

results = []
for item, answer, rethink, final_prompt in zip(records, generated_answers, rethink_info, final_generation_prompts):
    predicted_letter = port_wmdp.extract_choice_from_answer(answer, item['choices'])
    predicted_index = ord(predicted_letter) - ord('A') if predicted_letter in CHOICE_LABELS else None
    is_correct = predicted_index == item['correct_answer_index'] if predicted_index is not None else False
    results.append({
        **item,
        'generated_answer': answer,
        'generated_choice_letter': predicted_letter,
        'predicted_index': predicted_index,
        'is_correct': bool(is_correct),
        'rethink_triggered': bool(rethink),
        'generation_prompt': final_prompt,
    })

accuracy = sum(1 for row in results if row['is_correct']) / len(results)
valid_rate = sum(1 for row in results if row['predicted_index'] is not None) / len(results)
rethink_rate = sum(1 for row in results if row['rethink_triggered']) / len(results)
metrics = {
    'accuracy': accuracy,
    'valid_predictions_rate': valid_rate,
    'rethink_count': int(rethink_count),
    'rethink_rate': rethink_rate,
    'model_load_seconds': model_load_seconds,
    'run_seconds': run_seconds,
    'rows': len(results),
}

output_dir = Path(args.output_dir) / MODEL_NAME.replace('/', '_') / WMDP_SET
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'final_generations_full.json').write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
(output_dir / 'final_metrics_full.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(output_dir / 'rethink_stats.json').write_text(json.dumps({'test': int(rethink_count)}, indent=2), encoding='utf-8')
(output_dir / 'timing_stats.json').write_text(json.dumps({'model_load_seconds': model_load_seconds, 'test': run_seconds, 'pipeline_total_time': model_load_seconds + run_seconds}, indent=2), encoding='utf-8')
pd.DataFrame(results).to_csv(output_dir / 'predictions.csv', index=False)

summary_payload = {'run_config': run_config, 'metrics': metrics, 'output_dir': str(output_dir)}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')
print(json.dumps(summary_payload, indent=2, default=str)[:4000])


## Verify Smoke Artifacts

In [ ]:
output_dir = Path(args.output_dir) / MODEL_NAME.replace('/', '_') / WMDP_SET
required_files = [
    RUN_DIR / 'artifact_audit.json',
    RUN_DIR / 'run_config.json',
    RUN_DIR / 'summary.json',
    output_dir / 'final_generations_full.json',
    output_dir / 'final_metrics_full.json',
    output_dir / 'rethink_stats.json',
    output_dir / 'timing_stats.json',
    output_dir / 'predictions.csv',
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise RuntimeError(f'Missing expected artifacts: {missing_files}')
if len(results) != len(records):
    raise RuntimeError(f'Expected {len(records)} results, got {len(results)}')
if not (0.0 <= metrics['valid_predictions_rate'] <= 1.0):
    raise RuntimeError(f'Invalid valid_predictions_rate: {metrics["valid_predictions_rate"]}')

print('PAPER PORT WMDP SMOKE TEST COMPLETED')
print('Rows:', len(results))
print('Metrics:', json.dumps(metrics, indent=2))
print('Artifacts:', output_dir)
